In [ ]:
import importlib
from torch.utils.data import DataLoader, random_split
import torch

In [3]:
from functions_library import *

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [10]:
base_string = "C://Users//jonas//Desktop//Masters//Main//h5//"
data_strings = ["MC_EGAM1", "MC_EGAM7", "DATA_EGAM"]
name_strings = data_strings + ["MC_EGAM"]
full_strings = [base_string + data_string for data_string in data_strings]
h5_files_dir = data_loading.create_path_lists(full_strings, outputs=name_strings, mixing_list = [(0, 1)], seed=0)
h5_files_egam = h5_files_dir[name_strings[-1]]
h5_files_data = h5_files_dir[name_strings[-2]]

In [ ]:
model, train_losses, val_losses = model_training.compressed_training_function(
    h5_files = h5_files_egam,
    mix_files = h5_files_data,
    mixture_ratio = 0.8,
    dataset_class = dataset_classes.ZPairDataset,
    nn_model = neural_models.ParticlePairNetwork,
    model_kwargs = {"particle_hidden_sizes": [1024, 256, 256, 256, 256, 256, 64, 8], "event_hidden_sizes" : [64, 64, 16, 4], "consolidation_hidden_sizes" : [32, 32, 32, 8], "output_size" : 4,},
    loss_function = training_classes.GaussianNLLLoss(),
    optimizer_class = torch.optim.Adam,
    batch_size = 4096,
    num_workers = 8,
    exclude_features = ["truth"],
    exclude_fields = ["pt", "truthOrigin", "truthType"],
    fallback_source = "egammas",
    fallback_field = ["pt"],
    y_source = "truth",
    y_field = ["pt"],
    domain_adversary = True,
    log_target = True,
    domain_loss_function = torch.nn.BCEWithLogitsLoss(),
    pin_memory = False,
    num_epochs = 5
)

Epoch 1/10, Training Loss: 2.2251, Validation Loss: 0.0943ze 1000000, 999424 samples processed
Epoch 2/10, Training Loss: -0.1583, Validation Loss: -0.5400
Epoch 3/10, Training Loss: -0.4763, Validation Loss: -0.6731


KeyboardInterrupt: 

In [ ]:
mean_dataset = dataset_classes.H5EgammaDataset(
    h5_files_egam,
    mix_files = h5_files_data,
    exclude_features = ["truth"],
    exclude_fields = ["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
    y_source = "egammas",
    y_field = "truthOrigin",
    fallback_field = ["passid_Medium"],
    mixture_ratio = 0.8,
)

means, stds = misc_functions.compute_mean_std(mean_dataset, sample_size=1000000, batch_size=4096, num_workers=8)

train_size_egam = int(0.75 * len(h5_files_egam))
train_size_data = int(0.75 * len(h5_files_data))

train_dataset = dataset_classes.ZPairDataset(
    h5_files_egam[:train_size_egam],
    mix_files = h5_files_data[:train_size_egam],
    exclude_features = ["truth"],
    exclude_fields = ["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
    y_source = "egammas",
    y_field = "truthOrigin",
    fallback_field = ["passid_Medium"],
    mixture_ratio = 0.8,
)

val_dataset = dataset_classes.ZPairDataset(
    h5_files_egam[train_size_egam:],
    mix_files = h5_files_data[train_size_data:],
    exclude_features = ["truth"],
    exclude_fields = ["truthOrigin", "truthType", "passid_VeryLoose", "passid_Loose", "passid_Medium", "passid_Tight"],
    y_source = "egammas",
    y_field = "truthOrigin",
    fallback_field = ["passid_Medium"],
    mixture_ratio = 0.8,
)

train_sampler = data_loading.ShuffledContiguousBatchSampler(len(train_dataset), batch_size=4096, dataset=train_dataset)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, prefetch_factor=2, persistent_workers=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4096, shuffle=False, num_workers=4, prefetch_factor=2, persistent_workers=True, pin_memory=True)

particle_fields = [field for field in train_dataset.fields if not field.startswith("eventwise_")]
event_fields = [field for field in train_dataset.fields if field.startswith("eventwise_")]
particle_size = len(particle_fields) // 2
if len(particle_fields) % 2 != 0:
    raise ValueError("Pair particle fields are not evenly split between the two particles.")

model = neural_models.ParticlePairNetwork(
    particle_size,
    len(event_fields),
    particle_hidden_sizes=[1024, 256, 256, 256, 256, 256, 64, 8],
    event_hidden_sizes=[64, 64, 16, 4],
    consolidation_hidden_sizes=[32, 32, 32, 8],
    output_size=1,
).to(device)

domain_adversary = neural_models.DomainAdversary(
    embedding_size=8*2+4,
    domain_count=1,
)
gradient_reversal = neural_models.GradientReversal(strength = 0.1)




optimizer = torch.optim.Adam(list(model.parameters()) + list(domain_adversary.parameters()), lr=0.001)
loss_function = torch.nn.BCEWithLogitsLoss()

train_losses, val_losses = model_training.train_model(model, optimizer, loss_function, train_loader, val_loader, num_epochs=5, device='cpu', means=means, stds=stds, binary_target=13, domain_adversary = domain_adversary, gradient_reversal = gradient_reversal, domain_loss_function = loss_function, domain_loss_weight = 0.5, mixed_target_loss_weight = 0.2)

c:\Users\jonas\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
torch.save(model.state_dict(), "models_library/temp.pth")